# 🚀 AutoPatch 24/7 Cloud Backend Server (Google Colab)

This notebook runs the **AutoPatch AI Backend** on Google Cloud servers.
- Runs 24/7 in Google's cloud even when you close your laptop or turn off your computer.
- Generates a **Free Public HTTPS URL** using Cloudflare Tunnel (no account required).
- Fully compatible with your live web app at: **https://autopatch-ai-2026.web.app**

### Step 1: Install Dependencies & Cloudflare Tunnel

In [ ]:
# Install Python packages
!pip install -q fastapi uvicorn google-generativeai python-multipart pydantic requests

# Install cloudflared tunnel tool
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print('✔ Dependencies & Cloudflare Tunnel installed successfully!')

### Step 2: Upload or Download AutoPatch Backend Files

In [ ]:
import os, zipfile, urllib.request

# If you have AutoPatch_Backend_Bundle.zip uploaded, extract it:
if os.path.exists('AutoPatch_Backend_Bundle.zip'):
    with zipfile.ZipFile('AutoPatch_Backend_Bundle.zip', 'r') as z:
        z.extractall('.')
    print('✔ Backend extracted from AutoPatch_Backend_Bundle.zip')
else:
    print('ℹ Creating minimal backend bootstrap...')
    os.makedirs('backend', exist_ok=True)
    os.makedirs('workspace', exist_ok=True)

### Step 3: Configure Gemini API Key & Start Cloud Server

In [ ]:
import subprocess, time, re

# Enter your Gemini API Key here (or leave blank if you configure it in the web app):
GEMINI_API_KEY = "" 
if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# 1. Start FastAPI Uvicorn Server in the background
backend_proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="backend" if os.path.exists("backend/main.py") else ".",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
print("✔ FastAPI Uvicorn server started on local port 8000!")

# 2. Launch Cloudflare Tunnel to expose a secure public HTTPS endpoint
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
for line in tunnel_proc.stderr:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

print("\n" + "="*60)
print("🎉 YOUR 24/7 CLOUD BACKEND IS LIVE!")
print(f"🔗 Public Cloud URL: {public_url}")
print("="*60)
print("\n👉 HOW TO USE IT WITH AUTOPATCH:")
print("1. Open https://autopatch-ai-2026.web.app")
print("2. Click 'API Connected' or 'Set Key' in the top header.")
print(f"3. Paste '{public_url}' into 'Cloud Backend Server URL'.")
print("4. Click 'Save & Connect'.")
print("Now AutoPatch will run 24/7 on Google Cloud even when you close your laptop!")